# Union-of-Features Multi-Domain Neural Network

This notebook implements the larger-model idea we discussed.

Instead of using only the intersection of VED and OBD2 features, this notebook uses the **union** of features from both domains and trains a **multi-domain neural network**.

The design goal is:

- keep rich hybrid-engine information from the VED dataset
- allow diesel OBD2 rows to contribute using whatever signals they actually contain
- avoid forcing diesel rows to behave as if they had hybrid battery data
- support fuel-rate prediction across domains
- support SOC prediction only where SOC is meaningful and available

## Why a union model can help

The earlier shared-feature notebook used only the overlap between VED and OBD2. That was scientifically clean, but it threw away many useful VED features and left the model with only a tiny shared signal.

This notebook takes a different view:

- the **union of features** is allowed
- each sample carries a **feature-availability mask** telling the model which values are real and which are missing
- each sample also carries a **domain indicator** (`VED` or `OBD2`)
- the network has:
  - a small VED-specific encoder
  - a small OBD2-specific encoder
  - a shared backbone
  - a fuel-rate head for all domains
  - an SOC head only supervised on VED rows

This is closer to how a real general-purpose model should be built: not by pretending all datasets look the same, but by learning from their structure while respecting their differences.

## Important expectation setting

A bigger model does not automatically guarantee better performance. If the diesel dataset still contains only limited engine signals, the model may still struggle. But this architecture is a better test of the idea because it:

- preserves hybrid-only features instead of discarding them
- uses missing-feature masks explicitly
- lets each domain keep some dataset-specific representation
- avoids asking the SOC target to exist for diesel rows

In other words, this notebook is a stronger machine-learning design, even if the datasets still impose a ceiling on accuracy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import copy
import glob
import json
import math
import os
import random
import re
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

VED_GLOB = '/content/drive/MyDrive/Data/VED-dataset/VED_HEV_PHEV/*.csv'
OBD2_GLOB = '/content/drive/MyDrive/Data/OBD2-datalogger/Fiesta-TD-Ci/*.csv'

OUTPUT_DIR = '/content/drive/MyDrive/Data/VED-dataset/union_multidomain_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS = 20
BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
FUEL_LOSS_WEIGHT = 1.0
SOC_LOSS_WEIGHT = 0.4
DOMAIN_VED = 0
DOMAIN_OBD2 = 1


In [ ]:
def normalize_name(name):
    name = str(name).strip().lower()
    name = name.replace('[', ' ').replace(']', ' ')
    name = name.replace('(', ' ').replace(')', ' ')
    name = name.replace('%', ' percent ')
    name = name.replace('/', ' ')
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return re.sub(r'_+', '_', name).strip('_')

def resolve_alias(columns, aliases):
    normalized = {normalize_name(col): col for col in columns}
    for alias in aliases:
        found = normalized.get(normalize_name(alias))
        if found is not None:
            return found
    return None

def identity(series):
    return pd.to_numeric(series, errors='coerce')

def fahrenheit_to_celsius(series):
    values = pd.to_numeric(series, errors='coerce')
    return (values - 32.0) * (5.0 / 9.0)

def mph_to_kmh(series):
    values = pd.to_numeric(series, errors='coerce')
    return values * 1.60934

def inhg_to_kpa(series):
    values = pd.to_numeric(series, errors='coerce')
    return values * 3.38639

def gal_per_hr_to_l_per_hr(series):
    values = pd.to_numeric(series, errors='coerce')
    return values * 3.78541

UNION_FEATURE_SPECS = [
    {
        'name': 'day_num',
        'ved_aliases': ['DayNum', 'Day Num'],
        'obd2_aliases': [],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'veh_id',
        'ved_aliases': ['VehId', 'Veh ID'],
        'obd2_aliases': [],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'trip_id',
        'ved_aliases': ['Trip'],
        'obd2_aliases': [],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'time_signal',
        'ved_aliases': ['Timestamp(ms)', 'Timestamp (ms)', 'Timestamp [ms]'],
        'obd2_aliases': ['Time (sec)', 'Time [sec]', 'Time'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'latitude_deg',
        'ved_aliases': ['Latitude[deg]', 'Latitude (deg)', 'Latitude'],
        'obd2_aliases': ['Latitude (deg)', 'Latitude'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'longitude_deg',
        'ved_aliases': ['Longitude[deg]', 'Longitude (deg)', 'Longitude'],
        'obd2_aliases': ['Longitude (deg)', 'Longitude'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'vehicle_speed_kmh',
        'ved_aliases': ['Vehicle Speed[km/h]', 'Vehicle speed[km/h]', 'Speed[km/h]'],
        'obd2_aliases': ['Vehicle speed (MPH)', 'Vehicle speed (km/h)', 'Vehicle speed'],
        'ved_transform': identity,
        'obd2_transform': mph_to_kmh,
    },
    {
        'name': 'engine_rpm',
        'ved_aliases': ['Engine RPM[RPM]', 'Engine RPM', 'RPM'],
        'obd2_aliases': ['Engine RPM (RPM)', 'Engine RPM', 'RPM'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'outside_air_temp_c',
        'ved_aliases': ['OAT[DegC]', 'OAT (DegC)', 'Outside Air Temperature[C]', 'Ambient Temperature[C]'],
        'obd2_aliases': ['Ambient air temperature (degF)', 'Ambient air temperature (degC)', 'Ambient air temperature'],
        'ved_transform': identity,
        'obd2_transform': fahrenheit_to_celsius,
    },
    {
        'name': 'air_conditioning_power_kw',
        'ved_aliases': ['Air Conditioning Power[kW]', 'Air Conditioning Power (kW)'],
        'obd2_aliases': [],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'heater_power_watts',
        'ved_aliases': ['Heater Power[Watts]', 'Heater Power (Watts)'],
        'obd2_aliases': [],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'hv_battery_current_a',
        'ved_aliases': ['HV Battery Current[A]', 'HV Battery Current (A)'],
        'obd2_aliases': [],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'hv_battery_voltage_v',
        'ved_aliases': ['HV Battery Voltage[V]', 'HV Battery Voltage (V)'],
        'obd2_aliases': [],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'load_percent',
        'ved_aliases': ['Calculated load value[%]', 'Calculated load value (%)', 'Engine Load[%]', 'Load[%]'],
        'obd2_aliases': ['Calculated load value (%)', 'Calculated load value [%]', 'Engine load (%)'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'coolant_temp_c',
        'ved_aliases': ['Engine Coolant Temp[C]', 'Engine Coolant Temperature[C]', 'Coolant Temperature[C]'],
        'obd2_aliases': ['Engine coolant temperature (°F)', 'Engine coolant temperature (degF)', 'Engine coolant temperature'],
        'ved_transform': identity,
        'obd2_transform': fahrenheit_to_celsius,
    },
    {
        'name': 'manifold_pressure_kpa',
        'ved_aliases': ['Intake Manifold Absolute Pressure[kPa]', 'Manifold Absolute Pressure[kPa]', 'MAP[kPa]'],
        'obd2_aliases': ['Intake manifold absolute pressure (inHg)', 'Intake manifold absolute pressure (kPa)', 'MAP (kPa)'],
        'ved_transform': identity,
        'obd2_transform': inhg_to_kpa,
    },
    {
        'name': 'instant_fuel_economy_mpg',
        'ved_aliases': [],
        'obd2_aliases': ['Instant Fuel Economy (MPG)'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'total_fuel_economy_mpg',
        'ved_aliases': [],
        'obd2_aliases': ['Total Fuel Economy (MPG)'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'instant_co2_rate_lb_per_mile',
        'ved_aliases': [],
        'obd2_aliases': ['Instant CO2 rate (lb/mile)'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'trip_distance_miles',
        'ved_aliases': [],
        'obd2_aliases': ['Trip Distance (miles)'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'trip_duration_min',
        'ved_aliases': [],
        'obd2_aliases': ['Trip Duration (min)'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
]

VED_FUEL_ALIASES = ['Fuel Rate[L/hr]', 'Fuel Rate [L/hr]', 'Fuel Rate (L/hr)']
VED_SOC_ALIASES = ['HV Battery SOC[%]', 'HV Battery SOC [%]', 'HV Battery SOC (%)']
OBD2_FUEL_ALIASES = ['Fuel Rate (gal/hr)', 'Fuel Rate [gal/hr]']

def read_obd2_files(obd2_glob):
    files = sorted(glob.glob(obd2_glob))
    if not files:
        raise FileNotFoundError(f'No OBD2 CSV files found for {obd2_glob}')

    frames = []
    for file in files:
        frame = pd.read_csv(file, skiprows=1, encoding='utf-8-sig', skipinitialspace=True, on_bad_lines='skip')
        frame['source_file'] = Path(file).name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

def get_ved_files(ved_glob):
    files = sorted(glob.glob(ved_glob))
    if not files:
        raise FileNotFoundError(f'No VED CSV files found for {ved_glob}')
    return files

def resolve_union_specs(ved_columns, obd2_columns):
    resolved = []
    for spec in UNION_FEATURE_SPECS:
        item = dict(spec)
        item['ved_col'] = resolve_alias(ved_columns, spec['ved_aliases']) if spec['ved_aliases'] else None
        item['obd2_col'] = resolve_alias(obd2_columns, spec['obd2_aliases']) if spec['obd2_aliases'] else None
        if item['ved_col'] is not None or item['obd2_col'] is not None:
            resolved.append(item)
    return resolved

def build_ved_frame(files, specs, fuel_col, soc_col):
    required_cols = {fuel_col, soc_col}
    required_cols.update(item['ved_col'] for item in specs if item['ved_col'] is not None)
    frames = []

    for idx, file in enumerate(files, start=1):
        try:
            frame = pd.read_csv(file, usecols=lambda c: c in required_cols)
            if not frame.empty:
                frames.append(frame)
            if idx % 25 == 0:
                print(f'Loaded {idx} VED files...')
        except Exception as exc:
            print(f'Skipping {file} because of {type(exc).__name__}: {exc}')

    if not frames:
        raise ValueError('No VED frames were loaded.')

    raw = pd.concat(frames, ignore_index=True)
    out = pd.DataFrame(index=raw.index)
    mask = pd.DataFrame(index=raw.index)

    for item in specs:
        if item['ved_col'] is not None:
            values = item['ved_transform'](raw[item['ved_col']])
            out[item['name']] = values
            mask[item['name']] = values.notna().astype(float)
        else:
            out[item['name']] = np.nan
            mask[item['name']] = 0.0

    out['fuel_rate_lph'] = pd.to_numeric(raw[fuel_col], errors='coerce')
    out['soc_percent'] = pd.to_numeric(raw[soc_col], errors='coerce')
    out['dataset_name'] = 'VED'
    out['domain_id'] = DOMAIN_VED
    return out, mask

def build_obd2_frame(obd2_df, specs, fuel_col):
    out = pd.DataFrame(index=obd2_df.index)
    mask = pd.DataFrame(index=obd2_df.index)

    for item in specs:
        if item['obd2_col'] is not None:
            values = item['obd2_transform'](obd2_df[item['obd2_col']])
            out[item['name']] = values
            mask[item['name']] = values.notna().astype(float)
        else:
            out[item['name']] = np.nan
            mask[item['name']] = 0.0

    out['fuel_rate_lph'] = gal_per_hr_to_l_per_hr(obd2_df[fuel_col])
    out['soc_percent'] = np.nan
    out['dataset_name'] = 'OBD2'
    out['domain_id'] = DOMAIN_OBD2
    out['source_file'] = obd2_df['source_file']
    return out, mask

def chron_split(df, mask_df, train_ratio=0.8, val_ratio=0.1):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return (
        df.iloc[:train_end].reset_index(drop=True), mask_df.iloc[:train_end].reset_index(drop=True),
        df.iloc[train_end:val_end].reset_index(drop=True), mask_df.iloc[train_end:val_end].reset_index(drop=True),
        df.iloc[val_end:].reset_index(drop=True), mask_df.iloc[val_end:].reset_index(drop=True),
    )

def split_obd2_by_trip(df, mask_df):
    trips = sorted(df['source_file'].dropna().unique().tolist())
    if len(trips) < 2:
        raise ValueError('Need at least two OBD2 trip files for split.')
    test_trip = trips[-1]
    train_pool = df[df['source_file'] != test_trip].reset_index(drop=True)
    train_pool_mask = mask_df.loc[train_pool.index].reset_index(drop=True)
    test_df = df[df['source_file'] == test_trip].reset_index(drop=True)
    test_mask = mask_df.loc[test_df.index].reset_index(drop=True)
    cut = int(len(train_pool) * 0.8)
    return (
        train_pool.iloc[:cut].reset_index(drop=True), train_pool_mask.iloc[:cut].reset_index(drop=True),
        train_pool.iloc[cut:].reset_index(drop=True), train_pool_mask.iloc[cut:].reset_index(drop=True),
        test_df.reset_index(drop=True), test_mask.reset_index(drop=True),
        test_trip,
    )

class UnionDataset(Dataset):
    def __init__(self, features, avail_mask, domains, targets, target_mask):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.avail_mask = torch.tensor(avail_mask, dtype=torch.float32)
        self.domains = torch.tensor(domains, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.float32)
        self.target_mask = torch.tensor(target_mask, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.avail_mask[idx], self.domains[idx], self.targets[idx], self.target_mask[idx]

class UnionMultiDomainNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        combined_dim = input_dim * 2
        self.ved_encoder = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.obd2_encoder = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.shared = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
        )
        self.fuel_head = nn.Linear(16, 1)
        self.soc_head = nn.Linear(16, 1)

    def forward(self, x, avail_mask, domains):
        x_masked = x * avail_mask
        combined = torch.cat([x_masked, avail_mask], dim=1)
        ved_latent = self.ved_encoder(combined)
        obd2_latent = self.obd2_encoder(combined)
        domain_selector = (domains == DOMAIN_OBD2).float().unsqueeze(1)
        latent = ved_latent * (1.0 - domain_selector) + obd2_latent * domain_selector
        shared = self.shared(latent)
        fuel = self.fuel_head(shared)
        soc = self.soc_head(shared)
        return torch.cat([fuel, soc], dim=1)

def masked_mse(pred, target, target_mask):
    diff2 = (pred - target) ** 2
    weighted = diff2 * target_mask
    denom = target_mask.sum(dim=0).clamp_min(1.0)
    return weighted.sum(dim=0) / denom

def make_arrays(df, feature_cols, avail_mask_df):
    X = df[feature_cols].to_numpy(dtype=float)
    avail = avail_mask_df[feature_cols].to_numpy(dtype=float)
    domains = df['domain_id'].to_numpy(dtype=int)
    y = np.column_stack([
        df['fuel_rate_lph'].to_numpy(dtype=float),
        df['soc_percent'].to_numpy(dtype=float),
    ])
    target_mask = np.column_stack([
        np.isfinite(df['fuel_rate_lph'].to_numpy(dtype=float)).astype(float),
        np.isfinite(df['soc_percent'].to_numpy(dtype=float)).astype(float),
    ])
    return X, avail, domains, y, target_mask

def fit_model(model, train_loader, val_tensors, optimizer, epochs):
    best_state = None
    best_val = float('inf')
    history = {'train_total': [], 'val_total': [], 'train_fuel': [], 'train_soc': [], 'val_fuel': [], 'val_soc': []}
    vx, vm, vd, vy, vt = [t.to(device) for t in val_tensors]

    for epoch in range(epochs):
        model.train()
        running_total = 0.0
        running_fuel = 0.0
        running_soc = 0.0
        seen = 0

        for bx, bm, bd, by, bt in train_loader:
            bx = bx.to(device)
            bm = bm.to(device)
            bd = bd.to(device)
            by = by.to(device)
            bt = bt.to(device)

            optimizer.zero_grad()
            pred = model(bx, bm, bd)
            losses = masked_mse(pred, by, bt)
            total = FUEL_LOSS_WEIGHT * losses[0] + SOC_LOSS_WEIGHT * losses[1]
            total.backward()
            optimizer.step()

            batch_size = len(bx)
            running_total += total.item() * batch_size
            running_fuel += losses[0].item() * batch_size
            running_soc += losses[1].item() * batch_size
            seen += batch_size

        model.eval()
        with torch.no_grad():
            vpred = model(vx, vm, vd)
            vlosses = masked_mse(vpred, vy, vt)
            vtotal = FUEL_LOSS_WEIGHT * vlosses[0] + SOC_LOSS_WEIGHT * vlosses[1]

        train_total = running_total / max(seen, 1)
        train_fuel = running_fuel / max(seen, 1)
        train_soc = running_soc / max(seen, 1)

        history['train_total'].append(train_total)
        history['val_total'].append(vtotal.item())
        history['train_fuel'].append(train_fuel)
        history['train_soc'].append(train_soc)
        history['val_fuel'].append(vlosses[0].item())
        history['val_soc'].append(vlosses[1].item())

        if vtotal.item() < best_val:
            best_val = vtotal.item()
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | train_total={train_total:.4f} train_fuel={train_fuel:.4f} train_soc={train_soc:.4f} | "
            f"val_total={vtotal.item():.4f} val_fuel={vlosses[0].item():.4f} val_soc={vlosses[1].item():.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)
    return history

def eval_metrics(y_true, y_pred, mask, label):
    targets = ['fuel_rate_lph', 'soc_percent']
    for idx, name in enumerate(targets):
        valid = mask[:, idx] > 0
        if valid.sum() == 0:
            print(f'{label} | {name}: no valid targets')
            continue
        actual = y_true[valid, idx]
        pred = y_pred[valid, idx]
        mse = mean_squared_error(actual, pred)
        rmse = math.sqrt(mse)
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        print(f'{label} | {name}: MSE={mse:.4f} RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}')


## Resolve available union features

This cell tells us which features are available only in VED, which are available only in OBD2, and which are available in both. That summary is one of the most important outputs in this notebook.

In [ ]:
ved_files = get_ved_files(VED_GLOB)
ved_preview = pd.read_csv(ved_files[0], nrows=5)
ved_columns = list(ved_preview.columns)

obd2_raw = read_obd2_files(OBD2_GLOB)
obd2_columns = list(obd2_raw.columns)

resolved_specs = resolve_union_specs(ved_columns, obd2_columns)
feature_cols = [item['name'] for item in resolved_specs]

ved_only = [item['name'] for item in resolved_specs if item['ved_col'] is not None and item['obd2_col'] is None]
obd2_only = [item['name'] for item in resolved_specs if item['ved_col'] is None and item['obd2_col'] is not None]
shared = [item['name'] for item in resolved_specs if item['ved_col'] is not None and item['obd2_col'] is not None]

ved_fuel_col = resolve_alias(ved_columns, VED_FUEL_ALIASES)
ved_soc_col = resolve_alias(ved_columns, VED_SOC_ALIASES)
obd2_fuel_col = resolve_alias(obd2_columns, OBD2_FUEL_ALIASES)

if ved_fuel_col is None or ved_soc_col is None:
    raise ValueError('Could not resolve VED targets.')
if obd2_fuel_col is None:
    raise ValueError('Could not resolve OBD2 fuel target.')

print('Total union features resolved:', len(feature_cols))
print('Shared features:', shared)
print('VED-only features:', ved_only)
print('OBD2-only features:', obd2_only)


## Build VED and OBD2 frames with feature-availability masks

Each dataset row gets:

- the union feature vector
- a mask telling the model which features are really present
- a domain identifier (`VED` or `OBD2`)
- targets (`fuel`, and `SOC` only when available)

This lets the model use a larger feature space without pretending every row contains every feature.

In [ ]:
ved_df, ved_avail = build_ved_frame(ved_files, resolved_specs, ved_fuel_col, ved_soc_col)
obd2_df, obd2_avail = build_obd2_frame(obd2_raw, resolved_specs, obd2_fuel_col)

ved_keep = ved_df['fuel_rate_lph'].notna() & ved_df['soc_percent'].notna()
ved_df = ved_df.loc[ved_keep].reset_index(drop=True)
ved_avail = ved_avail.loc[ved_keep].reset_index(drop=True)

obd2_keep = obd2_df['fuel_rate_lph'].notna()
obd2_df = obd2_df.loc[obd2_keep].reset_index(drop=True)
obd2_avail = obd2_avail.loc[obd2_keep].reset_index(drop=True)

print('VED rows:', len(ved_df))
print('OBD2 rows:', len(obd2_df))


## Split the datasets and preprocess

We scale features after imputing missing values, but the network still sees the original feature-availability mask. That means the model is not relying only on imputed values; it also knows which values were actually observed.

In [ ]:
ved_train_df, ved_train_avail, ved_val_df, ved_val_avail, ved_test_df, ved_test_avail = chron_split(ved_df, ved_avail)
obd2_train_df, obd2_train_avail, obd2_val_df, obd2_val_avail, obd2_test_df, obd2_test_avail, held_out_trip = split_obd2_by_trip(obd2_df, obd2_avail)

print('VED split sizes:', len(ved_train_df), len(ved_val_df), len(ved_test_df))
print('OBD2 split sizes:', len(obd2_train_df), len(obd2_val_df), len(obd2_test_df))
print('Held-out OBD2 trip:', held_out_trip)

X_ved_train, A_ved_train, D_ved_train, y_ved_train, tm_ved_train = make_arrays(ved_train_df, feature_cols, ved_train_avail)
X_ved_val, A_ved_val, D_ved_val, y_ved_val, tm_ved_val = make_arrays(ved_val_df, feature_cols, ved_val_avail)
X_ved_test, A_ved_test, D_ved_test, y_ved_test, tm_ved_test = make_arrays(ved_test_df, feature_cols, ved_test_avail)

X_obd2_train, A_obd2_train, D_obd2_train, y_obd2_train, tm_obd2_train = make_arrays(obd2_train_df, feature_cols, obd2_train_avail)
X_obd2_val, A_obd2_val, D_obd2_val, y_obd2_val, tm_obd2_val = make_arrays(obd2_val_df, feature_cols, obd2_val_avail)
X_obd2_test, A_obd2_test, D_obd2_test, y_obd2_test, tm_obd2_test = make_arrays(obd2_test_df, feature_cols, obd2_test_avail)

imputer = SimpleImputer(strategy='constant', fill_value=0.0, keep_empty_features=True)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_ved_train_imp = imputer.fit_transform(X_ved_train)
X_ved_val_imp = imputer.transform(X_ved_val)
X_ved_test_imp = imputer.transform(X_ved_test)
X_obd2_train_imp = imputer.transform(X_obd2_train)
X_obd2_val_imp = imputer.transform(X_obd2_val)
X_obd2_test_imp = imputer.transform(X_obd2_test)

X_ved_train_scaled = scaler_X.fit_transform(X_ved_train_imp)
X_ved_val_scaled = scaler_X.transform(X_ved_val_imp)
X_ved_test_scaled = scaler_X.transform(X_ved_test_imp)
X_obd2_train_scaled = scaler_X.transform(X_obd2_train_imp)
X_obd2_val_scaled = scaler_X.transform(X_obd2_val_imp)
X_obd2_test_scaled = scaler_X.transform(X_obd2_test_imp)

y_ved_train_scaled = scaler_y.fit_transform(y_ved_train)
y_ved_val_scaled = scaler_y.transform(y_ved_val)
y_ved_test_scaled = scaler_y.transform(y_ved_test)

y_obd2_train_fill = y_obd2_train.copy()
y_obd2_val_fill = y_obd2_val.copy()
y_obd2_test_fill = y_obd2_test.copy()
soc_fill = float(np.nanmean(y_ved_train[:, 1]))
y_obd2_train_fill[:, 1] = soc_fill
y_obd2_val_fill[:, 1] = soc_fill
y_obd2_test_fill[:, 1] = soc_fill

y_obd2_train_scaled = scaler_y.transform(y_obd2_train_fill)
y_obd2_val_scaled = scaler_y.transform(y_obd2_val_fill)
y_obd2_test_scaled = scaler_y.transform(y_obd2_test_fill)

X_train = np.vstack([X_ved_train_scaled, X_obd2_train_scaled])
A_train = np.vstack([A_ved_train, A_obd2_train])
D_train = np.concatenate([D_ved_train, D_obd2_train])
y_train = np.vstack([y_ved_train_scaled, y_obd2_train_scaled])
tm_train = np.vstack([tm_ved_train, tm_obd2_train])

X_val = np.vstack([X_ved_val_scaled, X_obd2_val_scaled])
A_val = np.vstack([A_ved_val, A_obd2_val])
D_val = np.concatenate([D_ved_val, D_obd2_val])
y_val = np.vstack([y_ved_val_scaled, y_obd2_val_scaled])
tm_val = np.vstack([tm_ved_val, tm_obd2_val])

train_loader = DataLoader(UnionDataset(X_train, A_train, D_train, y_train, tm_train), batch_size=BATCH_SIZE, shuffle=True)
val_tensors = (
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(A_val, dtype=torch.float32),
    torch.tensor(D_val, dtype=torch.long),
    torch.tensor(y_val, dtype=torch.float32),
    torch.tensor(tm_val, dtype=torch.float32),
)


## Train the union multi-domain model

This network sees both the input values and the feature-availability mask. The domain-specific branch is selected by the row's domain ID, and then a shared backbone learns a latent representation used for prediction.

In [ ]:
model = UnionMultiDomainNet(input_dim=len(feature_cols)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = fit_model(model, train_loader, val_tensors, optimizer, EPOCHS)


## Evaluate on VED and OBD2 separately

This final section tells us whether the union approach helps us keep VED performance while also improving OBD2 fuel-rate behavior. That is the core question behind this experiment.

In [ ]:
def predict(x, a, d):
    xt = torch.tensor(x, dtype=torch.float32).to(device)
    at = torch.tensor(a, dtype=torch.float32).to(device)
    dt = torch.tensor(d, dtype=torch.long).to(device)
    with torch.no_grad():
        pred_scaled = model(xt, at, dt).cpu().numpy()
    return scaler_y.inverse_transform(pred_scaled)

ved_pred = predict(X_ved_test_scaled, A_ved_test, D_ved_test)
obd2_pred = predict(X_obd2_test_scaled, A_obd2_test, D_obd2_test)

print('Union feature count:', len(feature_cols))
print('Shared:', shared)
print('VED-only:', ved_only)
print('OBD2-only:', obd2_only)
print()

eval_metrics(y_ved_test, ved_pred, tm_ved_test, 'VED test')
print()
eval_metrics(y_obd2_test, obd2_pred, tm_obd2_test, 'OBD2 test')

plt.figure(figsize=(10, 4))
plt.plot(history['train_total'], label='Train total loss')
plt.plot(history['val_total'], label='Validation total loss')
plt.title('Union multi-domain training history')
plt.xlabel('Epoch')
plt.ylabel('Weighted loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(y_obd2_test[:, 0], label='Actual OBD2 fuel rate [L/hr]', linewidth=1.2)
plt.plot(obd2_pred[:, 0], label='Predicted OBD2 fuel rate [L/hr]', linewidth=1.2)
plt.title('OBD2 held-out trip fuel-rate prediction')
plt.xlabel('Sample index')
plt.ylabel('Fuel rate [L/hr]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(y_ved_test[:1000, 1], label='Actual VED SOC [%]', linewidth=1.2)
plt.plot(ved_pred[:1000, 1], label='Predicted VED SOC [%]', linewidth=1.2)
plt.title('VED SOC prediction sample')
plt.xlabel('Sample index')
plt.ylabel('SOC [%]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'union_multidomain_model.pth'))
joblib.dump(imputer, os.path.join(OUTPUT_DIR, 'union_imputer.pkl'))
joblib.dump(scaler_X, os.path.join(OUTPUT_DIR, 'union_scaler_X.pkl'))
joblib.dump(scaler_y, os.path.join(OUTPUT_DIR, 'union_scaler_y.pkl'))
with open(os.path.join(OUTPUT_DIR, 'union_feature_names.json'), 'w') as fh:
    json.dump(feature_cols, fh, indent=2)

obd2_out = obd2_test_df.copy()
obd2_out['predicted_fuel_rate_lph'] = obd2_pred[:, 0]
obd2_out.to_csv(os.path.join(OUTPUT_DIR, 'obd2_union_predictions.csv'), index=False)

print('Saved outputs to:', OUTPUT_DIR)
obd2_out.head()